# Defense Pipeline — 5 Attacks × 3 Defenses

**Hanium AML | 얼굴 인식 모델 적대적 공격 방어 평가**

---

## 실험 조건

공격 측이 생성한 적대적 이미지에 방어 기법 3종을 적용하고, ResNet-50 분류기의 예측 변화를 측정합니다.

| 공격 | 파라미터 | 방어 대상 샘플 조건 |
|------|----------|--------------------|
| FGSM   | ε = 0.005 | `clean_correct = True` |
| PGD    | ε = 0.03, α = 0.003, steps = 10 | `success_on_clean = True` |
| SQUARE | ε = 0.05, max_queries = 300 | (두 조건 모두) |
| JSMA   | θ = 0.05, steps = 20, pixels_per_step = 200 | |
| ZOO    | ε = 0.05, max_queries = 2000, coords_per_iter = 128, lr = 0.02 | |

| 방어 기법 | 파라미터 | 동작 원리 |
|-----------|----------|----------|
| JPEG 압축 | quality = 75 | 고주파 perturbation을 손실 압축으로 제거 |
| Gaussian Blur | radius = 3 | 픽셀 단위 노이즈 평활화 |
| Bit-depth 축소 | bits = 4 | 색상 양자화로 미세한 perturbation 소거 |

**평가 지표**

| 지표 | 정의 |
|------|------|
| Defense Success Rate | 방어 후 공격이 더 이상 성공하지 못한 비율 |
| Recovery Rate | 방어 후 원래 레이블로 복원된 비율 |
| Confidence Drop | 목표 클래스 신뢰도 감소량 (방어 전 − 후) |

---

## 사전 준비 — Google Drive 파일 구조

```
내 드라이브/hanium-aml-defense/
  hanium_attack_outputs.zip          ← 공격 결과 zip (attack_index.csv + 적대적 이미지)
  face_resnet50_lfw10/
    best.pt                          ← 학습된 ResNet-50 체크포인트
```

zip 내부 구조:
```
hanium_attack_outputs/
  outputs/attacks/attack_index.csv
  outputs/attacks/fgsm_face/images/...
  outputs/attacks/pgd_face/images/...
  ...
```

---

## 소스 코드 구조

```
src/defenses/
  defense_jpeg.py        ← JPEG 압축 방어 (--quality, --attack-family 등)
  defense_smoothing.py   ← Gaussian blur 방어 (--radius)
  defense_bitdepth.py    ← Bit-depth 축소 방어 (--bits)
  summarize_defense.py   ← 결과 CSV 집계
  run_defenses.py        ← 방어 3종 일괄 실행 오케스트레이터
  plot_results.py        ← 결과 로드 / 집계 / 시각화 / 보고서 생성
```

---
## Step 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2. 경로 설정 ← ⚠️ 여기만 수정

`DRIVE_BASE` 를 본인의 Google Drive 경로로 맞춰주세요.

In [ ]:
# ─── 여기만 수정 ──────────────────────────────────────
DRIVE_BASE = "/content/drive/MyDrive/hanium-aml-defense"
# ──────────────────────────────────────────────────────

ATTACK_ZIP_SRC    = f"{DRIVE_BASE}/hanium_attack_outputs.zip"
CKPT_SRC          = f"{DRIVE_BASE}/face_resnet50_lfw10/best.pt"
DRIVE_RESULTS_DIR = f"{DRIVE_BASE}/outputs/defenses"   # 결과 캐시 위치

# zip 압축 해제 후 경로 (수정 불필요)
ATTACK_ROOT = "/content/hanium_attack_outputs"
INDEX_PATH  = f"{ATTACK_ROOT}/outputs/attacks/attack_index.csv"

REPO_DIR    = "/content/26_HC160"
CKPT_DEST   = f"{REPO_DIR}/checkpoints/face_resnet50_lfw10/best.pt"
RESULTS_DIR = "/content/defense_results"
FIGURES_DIR = f"{RESULTS_DIR}/figures"

import os
for label, path in [("ATTACK_ZIP_SRC", ATTACK_ZIP_SRC), ("CKPT_SRC", CKPT_SRC)]:
    print(f"  {label}: {'OK' if os.path.exists(path) else '❌ 없음'} → {path}")

## Step 3. 레포 클론 + 패키지 설치

`src/defenses/` 하위 방어 스크립트와 `run_defenses.py`, `plot_results.py` 를 사용하기 위해 레포를 클론합니다.

In [ ]:
import os
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/Yaho03/26_HC160.git {REPO_DIR}
print(f"레포: {REPO_DIR}")

In [ ]:
!pip install -q torch torchvision pandas matplotlib seaborn tqdm

## Step 4. zip 압축 해제 및 체크포인트 확인

Drive에서 직접 이미지를 읽으면 느릴 수 있으니, zip은 Drive에 보관하고 실행할 때 `/content` 로컬 디스크에 풉니다.

In [ ]:
import shutil, os, pandas as pd
from pathlib import Path

# ── 1. zip → /content 로컬 디스크에 압축 해제 ─────────────
!mkdir -p /content/attack_outputs
!unzip -q {ATTACK_ZIP_SRC} -d /content/attack_outputs

# 실제 압축 해제된 디렉토리 구조 확인
!find /content/attack_outputs -maxdepth 4 -type d | head -20

# ── 2. attack_index.csv 위치 자동 탐색 ────────────────────
found = list(Path("/content/attack_outputs").rglob("attack_index.csv"))
assert found, (
    "attack_index.csv 를 찾지 못했습니다.\n"
    "위 find 출력을 확인해 zip 내부 구조를 파악하세요."
)

INDEX_PATH  = str(found[0])
# outputs/attacks/attack_index.csv 기준으로 3단계 위가 ATTACK_ROOT
ATTACK_ROOT = str(found[0].parent.parent.parent)
print(f"\nINDEX_PATH  : {INDEX_PATH}")
print(f"ATTACK_ROOT : {ATTACK_ROOT}")

# ── 3. 체크포인트 복사 ──────────────────────────────────────
assert os.path.exists(CKPT_SRC), (
    f"체크포인트 없음: {CKPT_SRC}\n"
    "Drive에 face_resnet50_lfw10/best.pt 를 업로드했는지 확인하세요."
)
os.makedirs(os.path.dirname(CKPT_DEST), exist_ok=True)
if not os.path.exists(CKPT_DEST):
    shutil.copy(CKPT_SRC, CKPT_DEST)
    print(f"체크포인트 복사: {CKPT_DEST}")
else:
    print(f"체크포인트 이미 존재: {CKPT_DEST}")

# ── 4. 방어 대상 샘플 수 확인 ──────────────────────────────
df_idx = pd.read_csv(INDEX_PATH)
ok = df_idx[(df_idx.clean_correct == True) & (df_idx.success_on_clean == True)]
print(f"\nattack_index.csv  전체 {len(df_idx):,}행 / 방어 대상 {len(ok):,}행")
print(ok.groupby('attack_family').size().rename('샘플수').to_string())

---
## Phase 1. 방어 기법 평가 실행

Drive에 이전 결과가 있으면 **재실행 없이 바로 로드**합니다.  
없을 경우에만 방어 3종을 실행하고, 완료 후 Drive에 자동 저장합니다.

```
Drive 캐시 있음 → /content 로 복사 후 바로 Phase 2
Drive 캐시 없음 → 방어 실행 → Drive에 저장 → Phase 2
```

In [ ]:
import sys, shutil, os
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Drive 캐시 파일 목록
CACHE_FILES = {
    "jpeg":      ("jpeg",      "jpeg_results_q75.csv"),
    "smoothing": ("smoothing", "smoothing_results_r3p0.csv"),
    "bitdepth":  ("bitdepth",  "bitdepth_results_4bit.csv"),
}

def _drive_cached() -> bool:
    """Drive에 결과 CSV 3개가 모두 있으면 True"""
    return all(
        os.path.exists(os.path.join(DRIVE_RESULTS_DIR, subdir, fname))
        for subdir, fname in CACHE_FILES.values()
    )

def _load_from_drive():
    """Drive 결과를 /content 로컬로 복사"""
    for subdir, fname in CACHE_FILES.values():
        src = os.path.join(DRIVE_RESULTS_DIR, subdir, fname)
        dst_dir = os.path.join(RESULTS_DIR, subdir)
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy(src, os.path.join(dst_dir, fname))
        print(f"  로드: {src}")

def _save_to_drive():
    """로컬 결과를 Drive에 저장"""
    for subdir, fname in CACHE_FILES.values():
        src = os.path.join(RESULTS_DIR, subdir, fname)
        dst_dir = os.path.join(DRIVE_RESULTS_DIR, subdir)
        os.makedirs(dst_dir, exist_ok=True)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(dst_dir, fname))
            print(f"  저장: {dst_dir}/{fname}")

# ── 캐시 확인 → 분기 ───────────────────────────────────────
if _drive_cached():
    print("✅ Drive에 저장된 결과 발견 → 재실행 생략\n")
    _load_from_drive()
    print("\n캐시 로드 완료")
else:
    print("⚙️  저장된 결과 없음 → 방어 기법 실행\n")
    from src.defenses.run_defenses import run_all_defenses
    run_all_defenses(
        attack_root = ATTACK_ROOT,
        index_path  = INDEX_PATH,
        ckpt_path   = CKPT_DEST,
        results_dir = RESULTS_DIR,
        repo_dir    = REPO_DIR,
    )
    print("\n💾 Drive에 결과 저장 중...")
    _save_to_drive()
    print("저장 완료")

### Phase 1 결과 파일 확인

In [ ]:
result_files = {
    "JPEG": f"{RESULTS_DIR}/jpeg/jpeg_results_q75.csv",
    "Smoothing": f"{RESULTS_DIR}/smoothing/smoothing_results_r3p0.csv",
    "Bitdepth": f"{RESULTS_DIR}/bitdepth/bitdepth_results_4bit.csv",
}
for name, path in result_files.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f"  {'OK' if exists else '❌'} [{name}] {path}  ({size:,} bytes)")

---
## Phase 2. 결과 요약

`src/defenses/summarize_defense.py` 로 방어 결과 CSV를 공격 × 방어 조합별로 집계합니다.  
출력 컬럼: `defense_success_rate`, `recovery_rate`, `avg_target_conf_drop`, `avg_defense_time_sec`

In [ ]:
import subprocess

SUMMARY_CSV = f"{RESULTS_DIR}/defense_summary.csv"
env = os.environ.copy()
env["PYTHONPATH"] = REPO_DIR + os.pathsep + env.get("PYTHONPATH", "")

subprocess.run(
    [sys.executable, "-m", "src.defenses.summarize_defense",
     "--defense-root", RESULTS_DIR,
     "--out", SUMMARY_CSV],
    cwd=ATTACK_ROOT, env=env, check=True,
)

### Phase 2. 상세 집계 테이블

`src/defenses/plot_results.py` 의 `load_results()` / `compute_summary()` 로
공격 × 방어 조합별 집계 테이블을 출력합니다.

In [ ]:
from src.defenses.plot_results import load_results, compute_summary

df      = load_results(RESULTS_DIR)
summary = compute_summary(df)

print(summary[[
    "attack_family", "defense_label", "n_samples",
    "defense_rate_%", "recovery_rate_%", "avg_conf_drop", "avg_time_ms"
]].to_string(index=False))

---
## Phase 3. 시각화

`src/defenses/plot_results.py` 의 `plot_all()` 이 3종 그래프를 생성합니다.

| 그래프 | 저장 파일 | 설명 |
|--------|-----------|------|
| Heatmap | `figures/heatmap.png` | 공격 × 방어 조합별 방어 성공률 / 복원율 |
| Bar chart | `figures/bar_by_attack.png` | 공격별 방어 성능 비교 (그룹 막대) |
| Box plot | `figures/boxplot_conf_drop.png` | 목표 클래스 신뢰도 감소 분포 |

In [ ]:
from src.defenses.plot_results import plot_all

plot_all(df, summary, FIGURES_DIR)

### Phase 3. 마크다운 보고서 생성

실험 조건 + 방어 성공률 / 복원율 / 신뢰도 감소 테이블을 `defense_report.md` 로 저장하고 인라인 렌더링합니다.

In [ ]:
from src.defenses.plot_results import generate_report

generate_report(summary, RESULTS_DIR)

---
## (선택) 그래프 · 보고서를 Drive에 백업

결과 CSV는 Phase 1에서 자동 저장됩니다.  
그래프와 마크다운 보고서를 추가로 백업하려면 이 셀을 실행하세요.

In [ ]:
import shutil, os

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

for fname in [
    "defense_summary.csv",
    "defense_report.md",
    "figures/heatmap.png",
    "figures/bar_by_attack.png",
    "figures/boxplot_conf_drop.png",
]:
    src = os.path.join(RESULTS_DIR, fname)
    dst = os.path.join(DRIVE_RESULTS_DIR, fname)
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(src, dst)
        print(f"백업: {dst}")
    else:
        print(f"건너뜀: {src}")